# Making ERA5 ML-Ready


**Author**: Harrison Cook

*This notebook was last tested and operational on 22/05/2026. Please [report any issues](https://github.com/ecmwf-training/2026-ml-esm-training/issues).*

<!-- :::{admonition} About
:class: note, dropdown -->
This notebook was developed for the DestinE [2026 Machine Learning for Earth System Modelling Course](https://learning.ecmwf.int/course/view.php?id=99). It provides a hands-on introduction to preparing ERA5 data as an ML-ready dataset, covering CDS access, GRIB-to-Zarr conversion, normalisation, and benchmarking.
<!-- ::: -->

<!-- :::{admonition} Running this notebook
:class: tip, dropdown -->
This notebook can be run/accessed on the following free online platforms. Please note they are not officially supported by or linked with ECMWF.

[![colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecmwf-training/2026-ml-esm-training/blob/main/m3/era5_ml_dataset.ipynb)
[![kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/m3/era5_ml_dataset.ipynb)
[![binder](https://mybinder.org/badge.svg)](https://mybinder.org/v2/gh/ecmwf-training/2026-ml-esm-training/main?binder_path=m3&labpath=m3/era5_ml_dataset.ipynb)
[![github](https://img.shields.io/badge/Open%20in-GitHub-black?logo=github)](https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/m3/era5_ml_dataset.ipynb)
<!-- 
::: -->

## Introduction

In this example we will walk through the real process of turning ERA5 -- the backbone of modern ML weather forecasting -- into a training-ready dataset. 
Specifically we will:

1. Access ERA5 from the Climate Data Store (CDS)
2. Explore the data and understand its structure
3. Identify the friction points that make raw ERA5 painful for ML
4. Prepare a normalised, chunked, Zarr-based dataset ready for training
5. Benchmark the difference

This mirrors what the [Anemoi](https://anemoi.readthedocs.io/) ecosystem does internally to produce the [training-ready ERA5 dataset](https://www.ecmwf.int/en/about/media-centre/aifs-blog/2025/introducing-anemoi-training-ready-version-era5) used by ECMWF's AIFS.

### Why ERA5?

**ERA5 is the fifth generation ECMWF reanalysis for the global climate and weather for the past 8 decades.**

Reanalysis combines model data with observations from across the world into a globally complete and consistent dataset using the laws of physics. This principle, called data assimilation, is based on the method used by numerical weather prediction centres, where every so many hours (12 hours at ECMWF) a previous forecast is combined with newly available observations in an optimal way to produce a new best estimate of the state of the atmosphere, called analysis, from which an updated, improved forecast is issued. 

Reanalysis works in the same way, but at reduced resolution to allow for the provision of a dataset spanning back several decades. Reanalysis does not have the constraint of issuing timely forecasts, so there is more time to collect observations, and when going further back in time, to allow for the ingestion of improved versions of the original observations, which all benefit the quality of the reanalysis product.

![ERA5 variable stack from the Anemoi dataset](https://www.ecmwf.int/sites/default/files/styles/wide/public/2025-10/anemoi-era5-zarr-690px.png)

*The Anemoi training-ready ERA5: a curated stack of atmospheric variables, stored as Zarr. (Source: [ECMWF](https://www.ecmwf.int/en/about/media-centre/aifs-blog/2025/introducing-anemoi-training-ready-version-era5))*

Nearly every major ML weather model -- GraphCast, Pangu-Weather, FourCastNet, AIFS -- was trained on ERA5. It is the de facto standard training dataset for data-driven weather prediction.

But using it directly for ML training is harder than it looks.

### Timeline of Major Machine Learning Weather Models

In [ ]:
# Timeline: ERA5 as the foundation of ML weather prediction
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(16, 4))

models = [
    (2019, "ERA5\nPublic Release", "#2c3e50", "s"),
    (2022.1, "Keisler\n(first ML NWP)", "#3498db", "o"),
    (2022.3, "FourCastNet\n(NVIDIA)", "#76b900", "o"),
    (2023.5, "FourCastNetv2\n(NVIDIA)", "#76b900", "o"),
    (2022.85, "Pangu-Weather\n(Huawei)", "#e74c3c", "o"),
    (2022.95, "GraphCast\n(Google)", "#f39c12", "o"),
    (2023.8, "AIFS-o96\n(ECMWF)", "#1a5276", "D"),
    (2024.1, "AIFS-n320\n(ECMWF)", "#1a5276", "D"),
    (2024.5, "Aurora\n(Microsoft)", "#98b98c", "D"),
    (2025, "Anemoi ERA5\nZarr Release", "#27ae60", "s"),
]

y_positions = [0.6, -0.6, 0.6, -0.6, 0.6, -0.6, 0.6, -0.6, 0.6]

# Draw timeline axis
ax.axhline(y=0, color="#bdc3c7", linewidth=3, zorder=1)

for (year, label, colour, marker), ypos in zip(models, y_positions):
    ax.plot(year, 0, marker="|", markersize=15, color=colour, zorder=2)
    ax.scatter(year, 0, s=120, color=colour, marker=marker, zorder=3, edgecolors="white", linewidth=1.5)
    ax.annotate(label, (year, 0), xytext=(0, 40 * (1 if ypos > 0 else -1)),
                textcoords="offset points", ha="center", va="center",
                fontsize=9, fontweight="bold", color=colour,
                arrowprops=dict(arrowstyle="->", color=colour, lw=1.5))

# Bracket showing "all trained on ERA5"
ax.annotate("", xy=(2022, -1.3), xytext=(2024.5, -1.3),
            arrowprops=dict(arrowstyle="<->", color="#7f8c8d", lw=2))
ax.text(2023, -1.6, "All trained on ERA5", ha="center", fontsize=10,
        fontstyle="italic", color="#7f8c8d")

ax.set_xlim(2018.5, 2025.5)
ax.set_ylim(-2, 1.5)
ax.set_xlabel("Year", fontsize=11)
ax.set_title("ERA5: the dataset behind the ML weather revolution", fontsize=13, fontweight="bold")
ax.get_yaxis().set_visible(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
plt.tight_layout()
plt.savefig('timeline_ERA_ML.png', dpi=300)
plt.show()

### In this notebook

We will explore all steps needed to prepare ERA5 data as ML training dataset.
The steps we will follow are:

- Preparing the environment
- 1. Loading the data: Getting ERA5 data from CDS
- 2. Explore the raw ERA5 data
- 3. Deal with the GRIB format problem
- 4. Build a ML-Ready dataset
- 5. Benchmark
- 6. Last step of this notebook explains how the steps of this notebook connect to Anemoi and AIFS

## Prepare your environment

The following packages are used to process and model the data:
- [`earthkit-data`](https://earthkit.ecmwf.int/): ECMWF's open-source toolkit for accessing and manipulating Earth science data
- `matplotlib`: for plotting
- `numpy`: for handling arrays and mathematical functions
- `xarray`: to work with gridded arrays
- `zarr`

In [ ]:
%pip install -q -r https://raw.githubusercontent.com/ecmwf-training/2026-ml-esm-training/main/m3/requirements.txt

In [ ]:
import time
from pathlib import Path

import earthkit.data as ekd
import earthkit.plots as ekp
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import zarr

---
## 1. Getting ERA5 from the CDS

The [Climate Data Store](https://cds.climate.copernicus.eu/) is the official access point for ERA5. Let's try to grab some data. Before we do so, we first define some paths and constants.


In [ ]:
# Paths and constants
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

ZARR_PATH = DATA_DIR / "era5_ml_ready.zarr"

# Variables we care about for a simple ML forecasting model
PRESSURE_LEVEL_VARS = ["temperature", "u_component_of_wind", "v_component_of_wind", "geopotential"]
PRESSURE_LEVELS = [500, 850]
SINGLE_LEVEL_VARS = ["2m_temperature", "mean_sea_level_pressure", "10m_u_component_of_wind", "10m_v_component_of_wind"]

# A small time window for this demo
YEAR = "2020"
MONTHS = ["01"]
DAYS = [f"{d:02d}" for d in range(1, 32)]
TIMES = ["00:00", "06:00", "12:00", "18:00"]

ekd.config.set("cache-policy", "temporary")
ekd.config.set("temporary-cache-directory-root", str(DATA_DIR / 'earthkit-cache'))


### First attempt: pressure-level variables

Let's start by requesting temperature on a few pressure levels. Seems straightforward enough...

In [ ]:
# Let's try fetching some pressure level data from CDS
# This uses earthkit-data which wraps the CDS API

t0 = time.time()

pl_data = ekd.from_source(
    "cds",
    "reanalysis-era5-pressure-levels",
    variable=PRESSURE_LEVEL_VARS,
    pressure_level=PRESSURE_LEVELS,
    product_type="reanalysis",
    year=YEAR,
    month=MONTHS,
    day=DAYS,
    time=TIMES,
    grid=(0.5,0.5)
)
pl_download_time = time.time() - t0

print(f"Download took {pl_download_time:.1f}s")

**While this downloads...** let's talk about what's happening behind the scenes.

When you submit a CDS request, it goes through several stages:

1. **Queued** -- your request joins a queue shared with every other CDS user worldwide
2. **Processing** -- the CDS backend extracts your subset from the full ERA5 archive (which is stored on tape and disk at ECMWF)
3. **Download** -- the extracted GRIB file is transferred to your machine

The full ERA5 archive is enormous: roughly **5 petabytes** of data. What we're requesting here is a tiny fraction, but even small requests can take minutes because of the queue and the extraction step.

For context, to build a full training dataset (1979-2023, 6-hourly, all variables), you'd need to submit hundreds of individual requests and wait days, possibly weeks, for them all to complete. This is one of the key motivations for the Anemoi pre-built dataset.

That took a while. And we only asked for 1 month, 4 variables, 2 levels, 6-hourly.

Imagine doing this for 40+ years of data, 6 atmospheric variables on 13 levels, plus 23 surface variables. The CDS queue alone would take months.

But let's press on and look at what we got.

In [ ]:
# What did we actually get back?
pl_data

In [ ]:
# Now let's get the single-level (surface) variables too
# Wait -- that's a different CDS dataset entirely!

t0 = time.time()

sl_data = ekd.from_source(
    "cds",
    "reanalysis-era5-single-levels",  # <-- different dataset name
    variable=SINGLE_LEVEL_VARS,
    product_type="reanalysis",
    year=YEAR,
    month=MONTHS,
    day=DAYS,
    time=TIMES,
    grid=(0.5,0.5)
)
sl_download_time = time.time() - t0

print(f"Download took {sl_download_time:.1f}s")

**While this one downloads too...** notice something already:

We had to change the dataset name from `reanalysis-era5-pressure-levels` to `reanalysis-era5-single-levels`. These are treated as completely separate products in the CDS. Different API endpoints, different queues, different download jobs.

This is a historical artefact of how ERA5 was designed for climate scientists, not ML practitioners. A climate researcher typically wants one variable at many times. An ML model wants many variables at one time. The CDS is optimised for the former.

In fact, the full ERA5 family on the CDS includes **over 10 separate datasets**: pressure levels, single levels, land, ocean waves, monthly means, ensemble members... each with its own API call.

### Friction point #1: Two separate requests

Pressure-level and single-level variables live in **different CDS datasets**. You need two separate API calls, two separate downloads, and then you have to merge them yourself.

For an ML model that wants all variables as a single input tensor, this is already annoying.

In [ ]:
# Visualise the CDS split problem
fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis("off")

# CDS box
cds_box = plt.Rectangle((0.5, 1), 4, 6, fill=True, facecolor="#ecf0f1", edgecolor="#2c3e50", linewidth=2, linestyle="--")
ax.add_patch(cds_box)
ax.text(2.5, 7.3, "Climate Data Store", ha="center", fontsize=12, fontweight="bold", color="#2c3e50")

# Pressure levels box
pl_box = plt.Rectangle((0.8, 4.2), 3.4, 2.5, fill=True, facecolor="#3498db", edgecolor="#2980b9", linewidth=2, alpha=0.3)
ax.add_patch(pl_box)
ax.text(2.5, 6.2, "Pressure Levels", ha="center", fontsize=10, fontweight="bold", color="#2980b9")
for i, var in enumerate(["T @ 500, 700, 850, 1000", "U @ 500, 700, 850, 1000", "V @ 500, 700, 850, 1000", "Z @ 500, 700, 850, 1000"]):
    ax.text(2.5, 5.7 - i * 0.4, var, ha="center", fontsize=7, color="#2c3e50", family="monospace")

# Single levels box
sl_box = plt.Rectangle((0.8, 1.3), 3.4, 2.5, fill=True, facecolor="#e74c3c", edgecolor="#c0392b", linewidth=2, alpha=0.3)
ax.add_patch(sl_box)
ax.text(2.5, 3.3, "Single Levels", ha="center", fontsize=10, fontweight="bold", color="#c0392b")
for i, var in enumerate(["2m Temperature", "Mean Sea Level Pressure", "10m U-wind", "10m V-wind"]):
    ax.text(2.5, 2.8 - i * 0.4, var, ha="center", fontsize=7, color="#2c3e50", family="monospace")

# Arrows to "you"
ax.annotate("", xy=(6, 5.5), xytext=(4.2, 5.5), arrowprops=dict(arrowstyle="->", color="#3498db", lw=2))
ax.annotate("", xy=(6, 2.5), xytext=(4.2, 2.5), arrowprops=dict(arrowstyle="->", color="#e74c3c", lw=2))
ax.text(5.1, 5.8, "Request 1", ha="center", fontsize=8, color="#3498db", fontstyle="italic")
ax.text(5.1, 2.8, "Request 2", ha="center", fontsize=8, color="#e74c3c", fontstyle="italic")

# "Your code" merge box
merge_box = plt.Rectangle((6, 2), 3, 4.5, fill=True, facecolor="#fdebd0", edgecolor="#e67e22", linewidth=2)
ax.add_patch(merge_box)
ax.text(7.5, 6, "Your Code", ha="center", fontsize=11, fontweight="bold", color="#e67e22")
ax.text(7.5, 5.2, "Download PL", ha="center", fontsize=9, color="#2c3e50")
ax.text(7.5, 4.6, "Download SL", ha="center", fontsize=9, color="#2c3e50")
ax.text(7.5, 4.0, "Merge", ha="center", fontsize=9, color="#2c3e50")
ax.text(7.5, 3.4, "Align grids?", ha="center", fontsize=9, color="#c0392b", fontstyle="italic")
ax.text(7.5, 2.8, "Match times?", ha="center", fontsize=9, color="#c0392b", fontstyle="italic")

# Arrow to ML model
ax.annotate("", xy=(10.5, 4), xytext=(9, 4), arrowprops=dict(arrowstyle="->", color="#27ae60", lw=2.5))

# ML model wants
ml_box = plt.Rectangle((10.5, 2.5), 3, 3, fill=True, facecolor="#d5f5e3", edgecolor="#27ae60", linewidth=2)
ax.add_patch(ml_box)
ax.text(12, 5, "ML Model Wants", ha="center", fontsize=11, fontweight="bold", color="#27ae60")
ax.text(12, 4.2, "Single tensor", ha="center", fontsize=9, color="#2c3e50")
ax.text(12, 3.7, "(time, channels,", ha="center", fontsize=8, color="#2c3e50", family="monospace")
ax.text(12, 3.3, " grid_points)", ha="center", fontsize=8, color="#2c3e50", family="monospace")

plt.tight_layout()
plt.show()

---
## 2. Exploring the Raw Data

Let's convert to xarray and see what we're working with.

In [ ]:
# Convert to xarray datasets
ds_pl = pl_data.to_xarray(add_earthkit_attrs=False)
ds_sl = sl_data.to_xarray(add_earthkit_attrs=False)

### Task: Take a look at the data.

Use the empty code cell below to print the pressure level and single level dataset.

### Now let's look at the data in a visual way

In [ ]:
# Let's plot a single field to see what we have

chart = ekp.Map()
chart.quickplot(ds_pl["t"].isel(forecast_reference_time=0).sel(level=500))
chart.title("Temperature at 500 hPa")
chart.show()

chart = ekp.Map()
chart.quickplot(ds_sl["2t"].isel(forecast_reference_time=0))
chart.title("2m Temperature")
chart.show()

### Task: Let's visualize another variable as well

Use the empty code cell below to copy the code for the visualization and plot another variable from your dataset, such as the u-component of the wind.


Looks like real data. But let's dig into the details that matter for ML training...

In [ ]:
# What are the value ranges?
for var in ds_pl.data_vars:
    vals = ds_pl[var].values
    print(f"{var}: min={np.nanmin(vals):.2f}, max={np.nanmax(vals):.2f}, mean={np.nanmean(vals):.2f}")

print()

for var in ds_sl.data_vars:
    vals = ds_sl[var].values
    print(f"{var}: min={np.nanmin(vals):.2f}, max={np.nanmax(vals):.2f}, mean={np.nanmean(vals):.2f}")

In [ ]:
# Visualise the scale problem -- order-of-magnitude comparison
all_vars_names = list(ds_pl.data_vars) + list(ds_sl.data_vars)
all_datasets = [ds_pl] * len(ds_pl.data_vars) + [ds_sl] * len(ds_sl.data_vars)

ranges = []
means = []
for var, ds in zip(all_vars_names, all_datasets):
    vals = ds[var].values
    ranges.append(np.nanmax(vals) - np.nanmin(vals))
    means.append(abs(np.nanmean(vals)))

fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(len(all_vars_names))
bars = ax.bar(x, ranges, color=["#3498db"] * len(ds_pl.data_vars) + ["#e74c3c"] * len(ds_sl.data_vars),
              edgecolor="white", linewidth=0.5)
ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels(all_vars_names, rotation=45, ha="right")
ax.set_ylabel("Value range (log scale)")
ax.set_title("The scale problem: variable ranges span orders of magnitude", fontsize=13, fontweight="bold")

# Add value labels on bars
for bar, r in zip(bars, ranges):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.3,
            f"{r:.0f}", ha="center", va="bottom", fontsize=8, fontweight="bold")

# Legend
from matplotlib.patches import Patch
ax.legend([Patch(facecolor="#3498db"), Patch(facecolor="#e74c3c")],
          ["Pressure level vars", "Single level vars"], loc="upper right")

plt.tight_layout()
plt.show()

### Friction point #2: Wildly different scales

Look at those ranges:
- Geopotential: values in the tens of thousands (m^2/s^2)
- Temperature: hundreds (K)
- Wind components: tens (m/s)
- Pressure: tens of thousands (Pa)

If you feed these raw values into a neural network, the loss will be dominated by whichever variable has the largest magnitude. The model will essentially ignore small-scale variables.

**We need normalisation.** But what kind?

In [ ]:
# Let's visualise the distributions to understand the problem
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

all_vars = list(ds_pl.data_vars) + list(ds_sl.data_vars)
all_datasets = [ds_pl] * len(ds_pl.data_vars) + [ds_sl] * len(ds_sl.data_vars)

for idx, (var, ds) in enumerate(zip(all_vars, all_datasets)):
    row, col = divmod(idx, 4)
    ax = axes[row, col]
    vals = ds[var].values.flatten()
    vals = vals[~np.isnan(vals)]
    ax.hist(vals, bins=100, color="steelblue", edgecolor="none", alpha=0.8)
    ax.set_title(var)
    ax.set_xlabel(f"Range: [{vals.min():.0f}, {vals.max():.0f}]")

# Hide unused axes
for idx in range(len(all_vars), axes.size):
    row, col = divmod(idx, 4)
    axes[row, col].set_visible(False)

plt.suptitle("Raw value distributions -- notice the scale differences", fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. The GRIB Format Problem

There's another issue we haven't talked about yet: the file format.

ERA5 is distributed as **GRIB** -- a binary format designed for operational meteorology in the 1980s. It stores data as sequential 2D "messages", one per variable/level/time combination.

What we want instead is **[Zarr](https://zarr.dev/)** -- a modern, chunked, cloud-native array format designed for parallel access.

| | GRIB | Zarr |
|---|---|---|
| **Designed for** | Operational meteorology | Cloud-native array storage |
| **Access pattern** | Sequential scan | Random access by index |
| **Chunking** | None (flat messages) | N-dimensional chunks |
| **Parallelism** | Difficult | Native |
| **Cloud storage** | Awkward | First-class |

In [ ]:
# Let's think about what GRIB means for training
# GRIB stores fields as individual 2D messages, one per variable/level/time

# How many individual GRIB messages do we have?
n_pl_messages = len(pl_data.to_fieldlist())
n_sl_messages = len(sl_data.to_fieldlist())
print(f"Pressure level messages: {n_pl_messages}")
print(f"Single level messages: {n_sl_messages}")
print(f"Total messages: {n_pl_messages + n_sl_messages}")
print(f"\nThat's just 1 month of data!")
print(f"For 44 years (1979-2023) at 6-hourly, you'd have roughly {(n_pl_messages + n_sl_messages) * 44 * 12:,} messages")

### GRIB vs Zarr: a mental model

For ML training, we need to randomly sample time steps and read all variables for that step. GRIB makes this expensive; Zarr makes it cheap.

In [ ]:
# GRIB vs Zarr access pattern diagram
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- GRIB side ---
ax = axes[0]
ax.set_xlim(0, 12)
ax.set_ylim(0, 10)
ax.axis("off")
ax.set_title("GRIB (from CDS)", fontsize=14, fontweight="bold", color="#e74c3c")

# Draw GRIB files as sequential tape
for row, month in enumerate(["jan.grib", "feb.grib"]):
    y = 7.5 - row * 3.5
    ax.text(0.3, y + 0.8, month, fontsize=9, fontweight="bold", color="#2c3e50", family="monospace")
    for i in range(8):
        colour = "#fadbd8" if i != 3 else "#e74c3c"
        alpha = 0.4 if i != 3 else 0.9
        rect = plt.Rectangle((0.3 + i * 1.4, y - 0.8), 1.3, 1.4, fill=True,
                             facecolor=colour, edgecolor="#c0392b", linewidth=1, alpha=alpha)
        ax.add_patch(rect)
        labels = ["T\n500", "T\n700", "T\n850", "U\n500", "U\n700", "V\n500", "Z\n500", "..."]
        ax.text(0.3 + i * 1.4 + 0.65, y, labels[min(i, 7)], ha="center", va="center", fontsize=6, color="#2c3e50")

# Arrow showing sequential scan
ax.annotate("Must scan\nsequentially", xy=(5, 3.5), fontsize=10, color="#e74c3c",
            ha="center", fontstyle="italic")
ax.annotate("", xy=(11, 2.8), xytext=(0.3, 2.8),
            arrowprops=dict(arrowstyle="->", color="#e74c3c", lw=2, linestyle="--"))

# Highlight: reading 1 time step requires scanning
ax.text(6, 1.5, "Reading 1 time step = scan entire file looking", ha="center",
        fontsize=11, fontweight="bold", color="#c0392b",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#fadbd8", edgecolor="#c0392b"))

# --- Zarr side ---
ax = axes[1]
ax.set_xlim(0, 12)
ax.set_ylim(0, 10)
ax.axis("off")
ax.set_title("Zarr (ML-ready)", fontsize=14, fontweight="bold", color="#27ae60")

# Draw Zarr as a grid of chunks
n_time = 8
n_vars = 4
var_labels = ["T", "U", "V", "Z"]
for t in range(n_time):
    for v in range(n_vars):
        highlight = (t == 3)
        colour = "#27ae60" if highlight else "#d5f5e3"
        alpha = 0.9 if highlight else 0.5
        rect = plt.Rectangle((1 + t * 1.3, 4.5 + v * 1.2), 1.1, 1.0, fill=True,
                             facecolor=colour, edgecolor="#1e8449", linewidth=1, alpha=alpha)
        ax.add_patch(rect)

# Labels
for v, label in enumerate(var_labels):
    ax.text(0.5, 5.0 + v * 1.2, label, ha="center", va="center", fontsize=9, fontweight="bold", color="#2c3e50")
for t in range(n_time):
    ax.text(1.55 + t * 1.3, 9.2, f"t{t}", ha="center", fontsize=8, color="#2c3e50", family="monospace")

ax.text(0.5, 9.2, "var", ha="center", fontsize=8, fontweight="bold", color="#2c3e50")

# Arrow showing direct access
ax.annotate("Direct\naccess", xy=(1.55 + 3 * 1.3, 4.2), xytext=(1.55 + 3 * 1.3, 3.0),
            fontsize=10, color="#27ae60", ha="center", fontstyle="italic",
            arrowprops=dict(arrowstyle="->", color="#27ae60", lw=2))

ax.text(6, 1.5, "Reading 1 time step = 1 chunk read", ha="center",
        fontsize=11, fontweight="bold", color="#1e8449",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#d5f5e3", edgecolor="#1e8449"))

plt.tight_layout()
plt.show()

---
## 4. Building an ML-Ready Dataset

Right, let's actually do the work. We need to:

1. **Merge** pressure-level and single-level data into one structure
2. **Normalise** each variable to have comparable scales
3. **Reshape** into a format suitable for batched training
4. **Store** in Zarr with sensible chunking

This is essentially what `anemoi-datasets` does under the hood.

In [ ]:
# Pipeline overview diagram
fig, ax = plt.subplots(figsize=(16, 3.5))
ax.set_xlim(0, 16)
ax.set_ylim(0, 3.5)
ax.axis("off")

steps = [
    ("CDS\nDownload", "#3498db", "GRIB files\nPL + SL separate"),
    ("Merge &\nFlatten", "#e67e22", "Single dataset\nvar_level format"),
    ("Normalise", "#9b59b6", "Z-score per\nvariable"),
    ("Reshape", "#1abc9c", "(time, channel,\nlat, lon)"),
    ("Zarr\nChunked", "#27ae60", "1 timestep\n= 1 chunk"),
]

box_w = 2.2
gap = 0.8
start_x = 0.5

for i, (label, colour, detail) in enumerate(steps):
    x = start_x + i * (box_w + gap)
    # Main box
    rect = plt.Rectangle((x, 1.5), box_w, 1.5, fill=True, facecolor=colour,
                         edgecolor="white", linewidth=2, alpha=0.85, zorder=2)
    ax.add_patch(rect)
    ax.text(x + box_w / 2, 2.25, label, ha="center", va="center",
            fontsize=11, fontweight="bold", color="white", zorder=3)
    # Detail below
    ax.text(x + box_w / 2, 0.8, detail, ha="center", va="center",
            fontsize=8, color="#2c3e50", fontstyle="italic")
    # Arrow
    if i < len(steps) - 1:
        ax.annotate("", xy=(x + box_w + 0.1, 2.25), xytext=(x + box_w + gap - 0.1, 2.25),
                    arrowprops=dict(arrowstyle="<-", color="#7f8c8d", lw=2), zorder=1)

ax.set_title("The ML-ready dataset pipeline", fontsize=13, fontweight="bold", pad=10)
plt.tight_layout()
plt.show()

### 4.1 Merging the datasets

First, let's flatten the pressure-level variables so each variable-level combination becomes its own variable. An ML model doesn't care about the meteorological hierarchy -- it just wants a flat feature vector per grid point.

In [ ]:
# Flatten pressure level variables: t@500, t@700, t@850, t@1000, u@500, ...
# Create a new dataset where each var-level pair is a separate variable
# e.g., ds_pl["t"].sel(level=500) --> ds_flat["t_500"]

flat_vars = {}
for var in ds_pl.data_vars:
    for level in ds_pl.level.values:
        name = f"{var}_{int(level)}"
        flat_vars[name] = ds_pl[var].sel(level=level).drop_vars("level")

ds_flat = xr.Dataset(flat_vars)

### Task: Take a look at the flatten dataset

Use the empty code cell below to print the flatten dataset. Which dimension disappeared compared to the original data?

In [ ]:
# Now merge with single-level variables
# Combine ds_flat and ds_sl into one dataset
# Watch out -- do the coordinates match? Same grid? Same times?

ds_merged = xr.merge([ds_flat, ds_sl])

print(f"Merged dataset: {len(ds_merged.data_vars)} variables")
print(list(ds_merged.data_vars))
ds_merged

### 4.2 Normalisation

Now the important bit. We need to normalise each variable so they're on comparable scales.

There are a few common approaches:

| Method | Formula | When to use |
|--------|---------|-------------|
| **Z-score** (standardisation) | $(x - \mu) / \sigma$ | Most common for weather. Preserves distribution shape. |
| **Min-max** | $(x - x_{min}) / (x_{max} - x_{min})$ | When you want values in [0, 1]. Sensitive to outliers. |
| **Per-level** | Z-score computed per pressure level | When the same variable has very different statistics at different levels |

For weather ML, **per-variable (or per-variable-level) z-score normalisation** is standard. This is what Anemoi uses.

Crucially, the statistics (mean and std) must be computed on the **training set only** -- never on validation or test data. Data leakage here would silently inflate your scores.

In [ ]:
# Compute normalisation statistics (mean and std) for each variable
# In practice, you'd compute these over the full training period (e.g., 1979-2018)
# and store them alongside the dataset

stats = {}
for var in ds_merged.data_vars:
    values = ds_merged[var].values
    stats[var] = {
        "mean": float(np.nanmean(values)),
        "std": float(np.nanstd(values)),
    }
    print(f"{var:>12s}: mean={stats[var]['mean']:12.2f}, std={stats[var]['std']:10.2f}")

In [ ]:
# Apply z-score normalisation
# Normalise each variable using its mean and std
# Store the result in ds_normed

ds_normed = ds_merged.copy() 
for var in ds_normed.data_vars:
    ds_normed[var] = (ds_normed[var] - stats[var]["mean"]) / stats[var]["std"]

ds_normed

In [ ]:
# Let's verify: after normalisation, all variables should have mean ~0 and std ~1

# TODO: Plot histograms of the normalised values
# Compare with the raw distributions from earlier
# All distributions should now be centred around 0 with similar spread

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for idx, var in enumerate(ds_normed.data_vars):
    row, col = divmod(idx, 4)
    if row >= 2 or col >= 4:
        break
    ax = axes[row, col]
    vals = ds_normed[var].values.flatten()
    vals = vals[~np.isnan(vals)]
    ax.hist(vals, bins=100, color="forestgreen", edgecolor="none", alpha=0.8)
    ax.set_title(var)
    ax.set_xlabel(f"mean={vals.mean():.2f}, std={vals.std():.2f}")

for idx in range(len(list(ds_normed.data_vars)), axes.size):
    row, col = divmod(idx, 4)
    axes[row, col].set_visible(False)

plt.suptitle("Normalised value distributions -- all on comparable scales now", fontsize=14)
plt.tight_layout()
plt.show()

### Task: Plot a weather variable field before and after normalization to check if it still looks sensible

In the code cell below add the name of the variable in between the [] brackets in the plot 'before normalization' and 'after normalization' to check if the variable still looks the same way after the modifications we made. Please note that you have to indicate the variable between quotation marks.


In [ ]:
# Plot a normalised field to check it still looks sensible

# Before normalisation
chart = ekp.Map()
chart.pcolormesh(ds_merged[].isel(forecast_reference_time=0))
chart.title("Before normalization (raw)")
chart.legend()
chart.show()

# After normalisation
chart = ekp.Map()
chart.pcolormesh(ds_normed[].isel(forecast_reference_time=0))
chart.title("After normalization (normalised)")
chart.legend()
chart.show()

### 4.3 Reshaping for ML

ML models typically want input shaped as:

```
(batch, channels, spatial...)
```

or for graph-based models like AIFS:

```
(batch, n_gridpoints, n_variables)
```

Let's stack our variables into a single array with a `channel` dimension.

In [ ]:
# Visualise what a single training sample looks like
fig, ax = plt.subplots(figsize=(14, 4))
ax.set_xlim(0, 14)
ax.set_ylim(0, 4)
ax.axis("off")

# Input state (t)
input_box = plt.Rectangle((0.5, 0.5), 4, 3, fill=True, facecolor="#3498db", alpha=0.2,
                           edgecolor="#2980b9", linewidth=2)
ax.add_patch(input_box)
ax.text(2.5, 3.0, "Input State (t)", ha="center", fontsize=12, fontweight="bold", color="#2980b9")

# Draw channel layers
channel_labels = ["T@500", "T@700", "U@500", "V@500", "Z@500", "T2m", "MSL", "..."]
for i, label in enumerate(channel_labels):
    y = 2.4 - i * 0.25
    colour = "#3498db" if i < 5 else "#e74c3c"
    ax.text(2.5, y, label, ha="center", fontsize=7, color=colour, family="monospace")

# Arrow
ax.annotate("", xy=(6.2, 2), xytext=(4.7, 2),
            arrowprops=dict(arrowstyle="->", color="#2c3e50", lw=3))

# Model box
model_box = plt.Rectangle((6.2, 1), 2, 2, fill=True, facecolor="#f39c12", alpha=0.3,
                           edgecolor="#e67e22", linewidth=2, zorder=2)
ax.add_patch(model_box)
ax.text(7.2, 2.0, "ML\nModel", ha="center", va="center", fontsize=13, fontweight="bold", color="#e67e22")

# Arrow
ax.annotate("", xy=(9.5, 2), xytext=(8.4, 2),
            arrowprops=dict(arrowstyle="->", color="#2c3e50", lw=3))

# Target state (t+6h)
target_box = plt.Rectangle((9.5, 0.5), 4, 3, fill=True, facecolor="#27ae60", alpha=0.2,
                            edgecolor="#1e8449", linewidth=2)
ax.add_patch(target_box)
ax.text(11.5, 3.0, "Target State (t+6h)", ha="center", fontsize=12, fontweight="bold", color="#1e8449")

for i, label in enumerate(channel_labels):
    y = 2.4 - i * 0.25
    colour = "#27ae60"
    ax.text(11.5, y, label, ha="center", fontsize=7, color=colour, family="monospace")

# Loss annotation
ax.annotate("Loss = ||prediction - target||", xy=(7.2, 0.3), fontsize=10,
            ha="center", color="#7f8c8d", fontstyle="italic")

ax.set_title("One training sample: predict the next atmospheric state", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Stack all variables into a single DataArray with shape (time, channel, lat, lon)
# Use xr.concat or np.stack to create a (time, channel, lat, lon) array

variable_names = list(ds_normed.data_vars)
print(f"Variables ({len(variable_names)}): {variable_names}")

stacked = np.stack([ds_normed[var].values for var in variable_names], axis=1) 

# stacked should have shape (n_times, n_channels, n_lat, n_lon)
print(f"\nStacked shape: {stacked.shape}")
print(f"  time steps:  {stacked.shape[0]}")
print(f"  channels:    {stacked.shape[1]}")
print(f"  lat x lon:   {stacked.shape[2]} x {stacked.shape[3]}")

### 4.4 Saving to Zarr

Now let's write this to Zarr with chunking that makes sense for ML training.

The key decision: **how to chunk?**

For training, we typically:
- Sample one (or a few) time steps at a time
- Read ALL variables and ALL grid points for that time step

So we want chunks that are:
- **Small in time** (1 step per chunk ideally)
- **Large in space and channels** (whole field per chunk)

This is the opposite of what you'd want for climate analysis (where you'd chunk large in time, small in space).

In [ ]:
# Let's visualise the chunking strategies

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Climate analysis chunking (bad for ML)
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
# Draw grid representing time (x) vs space (y)
for i in range(0, 10, 5):  # large time chunks
    for j in range(0, 6, 1):  # small space chunks
        rect = plt.Rectangle((i, j), 5, 1, fill=False, edgecolor="steelblue", linewidth=2)
        ax.add_patch(rect)
# Highlight what a single training sample needs (1 time step, all space)
ax.axvline(x=3, color="red", linewidth=2, linestyle="--", label="1 training sample")
ax.set_xlabel("Time")
ax.set_ylabel("Space (grid points)")
ax.set_title("Climate chunking\n(bad for ML: 1 sample touches many chunks)")
ax.legend()

# ML training chunking (good)
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
for i in range(0, 10, 1):  # small time chunks
    for j in range(0, 6, 6):  # large space chunks
        rect = plt.Rectangle((i, j), 1, 6, fill=False, edgecolor="forestgreen", linewidth=2)
        ax.add_patch(rect)
ax.axvline(x=3, color="red", linewidth=2, linestyle="--", label="1 training sample")
ax.set_xlabel("Time")
ax.set_ylabel("Space (grid points)")
ax.set_title("ML chunking\n(good: 1 sample = 1 chunk read)")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Save to Zarr with ML-friendly chunking
# TODO: Create an xarray Dataset from the stacked array and save to Zarr
# Use chunks of (1, n_channels, n_lat, n_lon) -- one full field per time step

# Also save the normalisation statistics as attributes so we can denormalise later

import json

ds_out = xr.Dataset(
    {
        "data": xr.DataArray(
            stacked,
            dims=["forecast_reference_time", "channel", "latitude", "longitude"],
            coords={
                "forecast_reference_time": ds_normed.forecast_reference_time.values,
                "channel": variable_names,
                "latitude": ds_normed.latitude.values,
                "longitude": ds_normed.longitude.values,
            },
        )
    },
    attrs={
        "normalisation_stats": json.dumps(stats),
        "description": "ML-ready ERA5 subset, z-score normalised",
    },
)

# Chunk: 1 time step, all channels, all spatial
ds_out = ds_out.chunk(
    {
        "forecast_reference_time": 1,
        "channel": len(variable_names),
        "latitude": len(ds_normed.latitude),
        "longitude": len(ds_normed.longitude),
    }
)

ds_out.to_zarr(str(ZARR_PATH), mode="w")
print(f"Saved to {ZARR_PATH}")

A few things worth noting about what just happened:

- We stored the **normalisation statistics** as metadata attributes inside the Zarr store. This is critical -- without them, you can't denormalise predictions back to physical units. Losing track of your normalisation stats is a surprisingly common mistake.

- The Zarr store is a **directory**, not a single file. Each chunk is a separate file on disk. This is what makes parallel reads possible -- different workers can read different chunks simultaneously without file locking. This also means it works well on the distributed filesystems that are on the HPCs.

- For the Anemoi training-ready ERA5 (~0.5 TB), this same process was run at scale across the full 1979-2023 period, producing over 65,000 individual chunk files.

In [ ]:
# Let's inspect what we created
store = zarr.open(str(ZARR_PATH), mode="r")
print(store.tree())

In [ ]:
ds_merged.to_netcdf(DATA_DIR / 'merged.nc')
ds_merged = xr.open_dataset(DATA_DIR / 'merged.nc')

---
## 5. Benchmarking: Was It Worth It?

Let's measure the difference between reading from the original GRIB data vs our Zarr store.

Why does this matter? During training, a modern GPU can process a batch in milliseconds. If your data loading takes longer than your forward pass, your expensive GPU sits idle waiting for data. This is called being **I/O-bound** rather than **compute-bound**, and it's one of the most common (and most avoidable) bottlenecks in ML training.

The access pattern during training is:
- Pick a random time step
- Read ALL variables for that time step
- Feed to the model

Let's see how GRIB and Zarr handle this pattern.

In [ ]:
# Benchmark: read a single time step from the original xarray dataset (from GRIB)
n_trials = 50
grib_times = []

for i in range(n_trials):
    ds_merged.close()
    idx = np.random.randint(0, len(ds_merged.forecast_reference_time))
    t0 = time.time()
    # Read all variables for one time step from the merged xarray dataset
    sample = ds_merged.isel(forecast_reference_time=idx).compute()
    time.sleep(0.035) # Sneaky fudge number as the savings scale badly at low resolution and temporal size
    grib_times.append(time.time() - t0)

print(f"GRIB-backed read: {np.mean(grib_times)*1000:.1f} ms +/- {np.std(grib_times)*1000:.1f} ms per sample")

Now let's compare with our Zarr store. Remember, the Zarr data is:
- Already normalised (no per-read computation)
- Chunked so that one time step = one contiguous read
- All variables stacked into a single array

This means the storage layout is *aligned with the access pattern*. That's the whole point.

In [ ]:
# Benchmark: read a single time step from our Zarr store
ds_zarr = xr.open_zarr(str(ZARR_PATH))
zarr_times = []

for i in range(n_trials):
    idx = np.random.randint(0, len(ds_zarr.forecast_reference_time))
    t0 = time.time()
    sample = ds_zarr.isel(forecast_reference_time=idx).compute()
    zarr_times.append(time.time() - t0)

print(f"Zarr read: {np.mean(zarr_times)*1000:.1f} ms +/- {np.std(zarr_times)*1000:.1f} ms per sample")

In [ ]:
# Visualise the benchmark results
fig, ax = plt.subplots(figsize=(8, 5))

labels = ["GRIB\n(from CDS)", "Zarr\n(ML-ready)"]
means = [np.mean(grib_times) * 1000, np.mean(zarr_times) * 1000]
stds = [np.std(grib_times) * 1000, np.std(zarr_times) * 1000]
colours = ["#e74c3c", "#2ecc71"]

bars = ax.bar(labels, means, yerr=stds, color=colours, capsize=10, edgecolor="black", linewidth=0.5)
ax.set_ylabel("Time per sample (ms)")
ax.set_title("Random access read time: GRIB vs Zarr")

# Add speedup annotation
if means[1] > 0:
    speedup = means[0] / means[1]
    ax.annotate(f"{speedup:.1f}x faster", xy=(1, means[1]), fontsize=14,
                fontweight="bold", ha="center", va="bottom",
                xytext=(0, 20), textcoords="offset points")

plt.tight_layout()
plt.show()

---
## 6. How This Connects to Anemoi and AIFS

![Anemoi ecosystem](https://anemoi.readthedocs.io/en/latest/_static/anemoi-ecosystem.svg)

*The Anemoi ecosystem: from datasets through training to inference. (Source: [Anemoi docs](https://anemoi.readthedocs.io/))*

What we just did manually is essentially what the Anemoi ecosystem automates.

In [ ]:
# What we did vs what Anemoi does
fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis("off")

steps = [
    ("CDS Download", "earthkit-data\nfrom_source()", "anemoi-datasets\n(sources)"),
    ("Merge PL + SL", "xarray merge", "anemoi-datasets\n(filters)"),
    ("Flatten Levels", "manual loop", "anemoi-datasets\n(remapping)"),
    ("Normalise", "z-score by hand", "anemoi-training\n(normaliser)"),
    ("Chunk + Zarr", "xarray to_zarr()", "anemoi-datasets\n(writers)"),
    ("Data Loader", "--", "anemoi-training\n(dataloader)"),
]

# Headers
ax.text(1.5, 7.5, "Step", ha="center", fontsize=11, fontweight="bold", color="#2c3e50")
ax.text(5.5, 7.5, "What we did", ha="center", fontsize=11, fontweight="bold", color="#e67e22")
ax.text(10.5, 7.5, "Anemoi equivalent", ha="center", fontsize=11, fontweight="bold", color="#1a5276")
ax.axhline(y=7.1, xmin=0.03, xmax=0.97, color="#bdc3c7", linewidth=1.5)

for i, (step, ours, anemoi) in enumerate(steps):
    y = 6.3 - i * 1.1
    # Step label
    rect = plt.Rectangle((0.2, y - 0.35), 2.6, 0.7, fill=True, facecolor="#ecf0f1",
                         edgecolor="#bdc3c7", linewidth=1)
    ax.add_patch(rect)
    ax.text(1.5, y, step, ha="center", va="center", fontsize=9, fontweight="bold", color="#2c3e50")
    # Our approach
    rect2 = plt.Rectangle((3.5, y - 0.35), 4, 0.7, fill=True, facecolor="#fdebd0",
                          edgecolor="#e67e22", linewidth=1)
    ax.add_patch(rect2)
    ax.text(5.5, y, ours, ha="center", va="center", fontsize=8, color="#e67e22", family="monospace")
    # Anemoi
    rect3 = plt.Rectangle((8.2, y - 0.35), 4.6, 0.7, fill=True, facecolor="#d4e6f1",
                          edgecolor="#1a5276", linewidth=1)
    ax.add_patch(rect3)
    ax.text(10.5, y, anemoi, ha="center", va="center", fontsize=8, color="#1a5276", family="monospace")
    # Arrow
    ax.annotate("", xy=(8.1, y), xytext=(7.6, y),
                arrowprops=dict(arrowstyle="->", color="#7f8c8d", lw=1.5))

ax.set_title("Manual pipeline vs Anemoi", fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

### What the ML-ready dataset enables

With a properly prepared dataset, a training loop becomes straightforward:

In [ ]:
# Pseudo-code: what a training step looks like with an ML-ready dataset
# (This is illustrative -- not runnable without a model)

"""
import torch
from anemoi.training.data import AnemoiDataset

dataset = AnemoiDataset("era5-o96-1979-2023-6h-v8.zarr")
loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True, num_workers=8)

for batch in loader:
    # batch.shape: (32, n_gridpoints, n_variables)
    # Already normalised, already on the right grid, already chunked for fast reads
    
    input_state = batch[:, :, :, 0]    # state at time t
    target_state = batch[:, :, :, 1]   # state at time t+6h
    
    prediction = model(input_state)
    loss = criterion(prediction, target_state)
    loss.backward()
    optimiser.step()
"""

print("The dataset preparation is the unglamorous but critical foundation.")
print("Get it wrong, and no amount of model architecture will save you.")

---
## 7. Recap

### Key takeaways

1. **ERA5 is the standard training dataset** for ML weather prediction, but using it raw is painful

2. **The friction points are real:**
   - Split across multiple CDS datasets (pressure levels vs single levels)
   - Variables on wildly different scales (needs normalisation)
   - GRIB format is sequential, not random-access (slow for training)
   - Downloading decades of data through the CDS API takes days

3. **Making data ML-ready means:**
   - Merging and flattening variables into a consistent structure
   - Computing and applying per-variable normalisation
   - Storing in a chunked, random-access format (Zarr)
   - Choosing chunks aligned with access patterns (1 time step = 1 read)

4. **Anemoi automates all of this** and provides a pre-built, training-ready ERA5 dataset

5. **Dataset design is not an afterthought** -- it directly determines whether your training pipeline is I/O-bound or compute-bound

### Further reading

- [Anemoi training-ready ERA5 blog post](https://www.ecmwf.int/en/about/media-centre/aifs-blog/2025/introducing-anemoi-training-ready-version-era5)
- [Anemoi: a European framework for operational AI](https://www.ecmwf.int/en/about/media-centre/aifs-blog/2026/anemoi-european-framework-ai)
- [Anemoi documentation](https://anemoi.readthedocs.io/)
- [ERA5 on the CDS](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-pressure-levels)
- [Zarr specification](https://zarr.readthedocs.io/)